# Image Studio

A small Gradio app using the same ideas as day 5: custom Blocks UI, chat history, and image generation.

- First message **creates** an image from your prompt
- Later messages **edit that same image** (keep the monkey, change the background, etc.)
- Type `stop`, `done`, or `quit`, or click **Stop editing**, when you are finished
- Click **New image** to throw away the current picture and start again

Each generate/edit call costs a few cents — do not loop endlessly.

In [1]:
import os
import base64
from io import BytesIO

from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image
import gradio as gr

/Users/ravi1992/projects/generative-ai-projects/Image_Studio/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

openai = OpenAI()
IMAGE_MODEL = "gpt-image-1-mini"
IMAGE_SIZE = "1024x1024"

STOP_PHRASES = {
    "stop",
    "done",
    "quit",
    "exit",
    "finish",
    "no more",
    "that's enough",
    "thats enough",
    "stop editing",
}

OpenAI API Key not set


OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [3]:
def decode_image(b64_json):
    return Image.open(BytesIO(base64.b64decode(b64_json)))


def image_as_png_file(image):
    buf = BytesIO()
    image.convert("RGBA").save(buf, format="PNG")
    buf.seek(0)
    buf.name = "current.png"
    return buf


def generate_image(prompt):
    response = openai.images.generate(
        model=IMAGE_MODEL,
        prompt=prompt,
        size=IMAGE_SIZE,
        n=1,
    )
    return decode_image(response.data[0].b64_json)


def edit_image(image, prompt):
    response = openai.images.edit(
        model=IMAGE_MODEL,
        image=image_as_png_file(image),
        prompt=(
            "Edit this existing image. Keep the same main subject and overall look "
            f"unless the user asks to change them. User request: {prompt}"
        ),
        size=IMAGE_SIZE,
        n=1,
    )
    return decode_image(response.data[0].b64_json)


def wants_to_stop(text):
    return text.strip().lower() in STOP_PHRASES

In [4]:
def put_message_in_chatbot(message, history):
    history = history or []
    if not message or not str(message).strip():
        return message or "", history, "Type a prompt, then click Generate / Edit (or press Enter)."
    return "", history + [{"role": "user", "content": str(message).strip()}], "Working... image generation can take 20–40 seconds."


def generate_or_edit(history, current_image, stopped):
    history = history or []
    if not history:
        return history, current_image, current_image, stopped, "Type a prompt, then click Generate / Edit."

    user_text = history[-1]["content"]

    if stopped:
        history = history + [{
            "role": "assistant",
            "content": "Editing is stopped. Click New image to start a fresh picture.",
        }]
        return history, current_image, current_image, True, "Stopped. Click New image to start again."

    if wants_to_stop(user_text):
        history = history + [{
            "role": "assistant",
            "content": "Stopped. The last image is kept. Click New image when you want a new one.",
        }]
        return history, current_image, current_image, True, "Stopped. Last image kept."

    try:
        if current_image is None:
            image = generate_image(user_text)
            reply = "Created the image. Describe a change, then click Generate / Edit again. Or type stop."
            status = "New image generated. Keep chatting to edit it."
        else:
            image = edit_image(current_image, user_text)
            reply = "Updated the image from your last instruction. Keep going, or type stop."
            status = "Image edited from your last message."
    except Exception as e:
        history = history + [{
            "role": "assistant",
            "content": f"Image request failed: {e}",
        }]
        return history, current_image, current_image, stopped, f"Error: {e}"

    history = history + [{"role": "assistant", "content": reply}]
    return history, image, image, False, status


def stop_editing(history, current_image):
    history = (history or []) + [{
        "role": "assistant",
        "content": "Stopped. Click New image to start a fresh picture.",
    }]
    return history, current_image, True, "Stopped. Last image kept."


def new_image():
    return [], None, None, False, "Ready. Describe the first image, then click Generate / Edit."

In [5]:
with gr.Blocks(title="Image Studio") as ui:
    current_image = gr.State(None)
    stopped = gr.State(False)

    gr.Markdown(
        "## Image Studio\n"
        "Type a prompt and click **Generate / Edit** (or press Enter). "
        "The first prompt creates an image. Later prompts edit **that** image. "
        "Example: `funky monkey` then `change the background to a lake and trees where the monkey is sitting`."
    )

    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages", label="Instructions")
        image_output = gr.Image(height=500, interactive=False, label="Current image", type="pil")

    status = gr.Textbox(
        value="Ready. Describe the first image, then click Generate / Edit.",
        label="Status",
        interactive=False,
    )

    with gr.Row():
        message = gr.Textbox(
            label="Prompt",
            placeholder="funky monkey",
            scale=4,
            submit_btn=False,
        )
        send_btn = gr.Button("Generate / Edit", variant="primary", scale=1)

    with gr.Row():
        stop_btn = gr.Button("Stop editing")
        reset_btn = gr.Button("New image")

    send = send_btn.click(
        put_message_in_chatbot,
        inputs=[message, chatbot],
        outputs=[message, chatbot, status],
    ).then(
        generate_or_edit,
        inputs=[chatbot, current_image, stopped],
        outputs=[chatbot, image_output, current_image, stopped, status],
    )

    message.submit(
        put_message_in_chatbot,
        inputs=[message, chatbot],
        outputs=[message, chatbot, status],
    ).then(
        generate_or_edit,
        inputs=[chatbot, current_image, stopped],
        outputs=[chatbot, image_output, current_image, stopped, status],
    )

    stop_btn.click(
        stop_editing,
        inputs=[chatbot, current_image],
        outputs=[chatbot, current_image, stopped, status],
    )

    reset_btn.click(
        new_image,
        outputs=[chatbot, image_output, current_image, stopped, status],
    )

ui.queue()
ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
